# 01 — Pipeline de Features
Charge ou calcule toutes les features, les sauvegarde dans `data/processed_clean/`.
**Lancer une seule fois.** Toutes les cellules sont idempotentes (rien n'est recalculé si le cache existe).

In [17]:
import os, sys, pickle, shutil
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [18]:
BASE         = r'C:/Users/veyss/Nouveau dossier/nov25_alt1_mle_rakuten'
SPLIT_DIR    = os.path.join(BASE, 'data', 'split_no_crop')
OLD_PROC     = os.path.join(BASE, 'data', 'processed')
CLEAN_PROC   = os.path.join(BASE, 'data', 'processed_clean')
CLEAN_MODELS = os.path.join(BASE, 'models', 'clean')
os.makedirs(CLEAN_PROC,   exist_ok=True)
os.makedirs(CLEAN_MODELS, exist_ok=True)

def _copy(src, dst):
    if not os.path.exists(dst): shutil.copy2(src, dst)

print(f'CLEAN_PROC   : {CLEAN_PROC}')
print(f'CLEAN_MODELS : {CLEAN_MODELS}')

CLEAN_PROC   : C:/Users/veyss/Nouveau dossier/nov25_alt1_mle_rakuten\data\processed_clean
CLEAN_MODELS : C:/Users/veyss/Nouveau dossier/nov25_alt1_mle_rakuten\models\clean


In [19]:
train_split = pd.read_csv(os.path.join(SPLIT_DIR, 'train.csv'))
val_split   = pd.read_csv(os.path.join(SPLIT_DIR, 'val.csv'))
test_split  = pd.read_csv(os.path.join(SPLIT_DIR, 'test.csv'))

y_train = train_split['label'].values.astype(int)
y_val   = val_split['label'].values.astype(int)
y_test  = test_split['label'].values.astype(int)

# Sauvegarde labels + splits
for name, arr in [('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    np.save(os.path.join(CLEAN_PROC, f'{name}.npy'), arr)

for s in ['train', 'val', 'test']:
    _copy(os.path.join(SPLIT_DIR, f'{s}.csv'), os.path.join(CLEAN_PROC, f'{s}.csv'))
_copy(os.path.join(BASE, 'models', 'label_encoder.pkl'),
      os.path.join(CLEAN_PROC, 'label_encoder.pkl'))

# text_combined     : NLTK tokenisé, stopwords FR/EN/DE/ES supprimés → TF-IDF / c-TF-IDF
# text_combined_raw : brut après nettoyage HTML                       → Transformers (e5)
texts_clean = {
    'train': train_split['text_combined'].fillna('').tolist(),
    'val'  : val_split['text_combined'].fillna('').tolist(),
    'test' : test_split['text_combined'].fillna('').tolist(),
}
texts_raw = {
    'train': train_split['text_combined_raw'].fillna('').tolist(),
    'val'  : val_split['text_combined_raw'].fillna('').tolist(),
    'test' : test_split['text_combined_raw'].fillna('').tolist(),
}

print(f'Splits : train={len(y_train):,} | val={len(y_val):,} | test={len(y_test):,}')
print(f'Exemple text_combined     : {texts_clean["train"][2][:100]}')
print(f'Exemple text_combined_raw : {texts_raw["train"][2][:100]}')

Splits : train=59,185 | val=12,993 | test=12,738
Exemple text_combined     : cloche d'islande d'halldór kiljan laxness
Exemple text_combined_raw : la cloche d'islande d'halldór kiljan laxness


In [20]:
# ── TF-IDF (sur text_combined : tokenisé + stopwords supprimés) ───────
# NOTE : si le cache vient d'un run précédent avec text_combined_raw,
#        supprimer X_tfidf_*.npz dans processed_clean/ avant de relancer.
dst = os.path.join(CLEAN_PROC, 'X_tfidf_train.npz')
if os.path.exists(dst):
    print('TF-IDF : cache trouve')
else:
    print('TF-IDF : calcul sur text_combined (tokenise + stopwords supprimes)...')
    vec = TfidfVectorizer(max_features=50000, sublinear_tf=True, min_df=2, ngram_range=(1, 2))
    Xtr = vec.fit_transform(texts_clean['train'])
    sp.save_npz(os.path.join(CLEAN_PROC, 'X_tfidf_train.npz'), Xtr.tocsr())
    sp.save_npz(os.path.join(CLEAN_PROC, 'X_tfidf_val.npz'),   vec.transform(texts_clean['val']).tocsr())
    sp.save_npz(os.path.join(CLEAN_PROC, 'X_tfidf_test.npz'),  vec.transform(texts_clean['test']).tocsr())
    with open(os.path.join(CLEAN_MODELS, 'tfidf_vectorizer.pkl'), 'wb') as f: pickle.dump(vec, f)
    print(f'TF-IDF : {Xtr.shape}')

X_tfidf_tr = sp.load_npz(os.path.join(CLEAN_PROC, 'X_tfidf_train.npz'))
print(f'TF-IDF : {X_tfidf_tr.shape}')

TF-IDF : calcul sur text_combined (tokenise + stopwords supprimes)...
TF-IDF : (59185, 50000)
TF-IDF : (59185, 50000)


In [21]:
# ── c-TF-IDF (sur text_combined : tokenisé + stopwords supprimés) ────
# NOTE : si le cache vient d'un run précédent avec text_combined_raw,
#        supprimer X_ctfidf_*.npy dans processed_clean/ avant de relancer.
dst = os.path.join(CLEAN_PROC, 'X_ctfidf_train.npy')
if os.path.exists(dst):
    print('c-TF-IDF : cache trouve')
else:
    from sklearn.preprocessing import normalize as sp_norm
    print('c-TF-IDF : calcul mega-documents par classe (text_combined)...')
    mega = [''] * 27
    for t, l in zip(texts_clean['train'], y_train): mega[l] += ' ' + t
    cv = TfidfVectorizer(max_features=50000, sublinear_tf=True, min_df=1)
    C  = sp_norm(cv.fit_transform(mega), norm='l2')
    def _sim(texts_):
        X = sp_norm(cv.transform(texts_), norm='l2')
        return (X @ C.T).toarray().astype('float32')
    for s in ['train', 'val', 'test']:
        np.save(os.path.join(CLEAN_PROC, f'X_ctfidf_{s}.npy'), _sim(texts_clean[s]))
    print('c-TF-IDF : calcule')

X_ctfidf_tr = np.load(os.path.join(CLEAN_PROC, 'X_ctfidf_train.npy'))
print(f'c-TF-IDF : {X_ctfidf_tr.shape}')

c-TF-IDF : calcul mega-documents par classe (text_combined)...
c-TF-IDF : calcule
c-TF-IDF : (59185, 27)


In [22]:
# ── Embeddings e5-large (sur text_combined_raw : brut pour Transformer) ─
dst = os.path.join(CLEAN_PROC, 'X_e5_train.npy')
if os.path.exists(dst):
    print('e5 : cache trouve')
else:
    old = os.path.join(OLD_PROC, 'X_st_multilingual-e5-large_train.npy')
    if os.path.exists(old):
        for s in ['train', 'val', 'test']:
            _copy(os.path.join(OLD_PROC, f'X_st_multilingual-e5-large_{s}.npy'),
                  os.path.join(CLEAN_PROC, f'X_e5_{s}.npy'))
        print('e5 : copie depuis processed/')
    else:
        from sentence_transformers import SentenceTransformer
        print('e5 : encodage sur text_combined_raw (~5 min GPU)...')
        st = SentenceTransformer('intfloat/multilingual-e5-large', device=str(device),
                                  model_kwargs={'torch_dtype': 'float16'})
        for s in ['train', 'val', 'test']:
            arr = st.encode(texts_raw[s], batch_size=64, show_progress_bar=True,
                             normalize_embeddings=True, convert_to_numpy=True)
            np.save(os.path.join(CLEAN_PROC, f'X_e5_{s}.npy'), arr)
        print('e5 : encode')

X_e5_tr = np.load(os.path.join(CLEAN_PROC, 'X_e5_train.npy'))
print(f'e5-large : {X_e5_tr.shape}')

e5 : copie depuis processed/
e5-large : (59185, 1024)


In [23]:
# ── DINOv2 features (backbone gele) ──────────────────────────────────
dst = os.path.join(CLEAN_PROC, 'X_dino_train.npy')
if os.path.exists(dst):
    print('DINOv2 : cache trouve')
else:
    old = os.path.join(OLD_PROC, 'X_img_dinov2_train.npy')
    if os.path.exists(old):
        for s in ['train', 'val', 'test']:
            _copy(os.path.join(OLD_PROC, f'X_img_dinov2_{s}.npy'),
                  os.path.join(CLEAN_PROC, f'X_dino_{s}.npy'))
        print('DINOv2 : copie depuis processed/')
    else:
        from transformers import AutoModel
        from torch.utils.data import Dataset, DataLoader
        import torchvision.transforms as T
        from PIL import Image
        from tqdm import tqdm
        print('DINOv2 : extraction (~3 min GPU)...')
        dino = AutoModel.from_pretrained('facebook/dinov2-base').eval().to(device)
        for p in dino.parameters(): p.requires_grad = False
        transform = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
        class ImgDS(Dataset):
            def __init__(self, df, img_dir):
                self.df = df.reset_index(drop=True); self.img_dir = img_dir
            def __len__(self): return len(self.df)
            def __getitem__(self, i):
                try: return transform(Image.open(os.path.join(self.img_dir, self.df.iloc[i]['image_name'])).convert('RGB'))
                except: return torch.zeros(3, 224, 224)
        for s, df in [('train', train_split), ('val', val_split), ('test', test_split)]:
            loader = DataLoader(ImgDS(df, os.path.join(SPLIT_DIR, s)), batch_size=64, shuffle=False, num_workers=0)
            out = []
            with torch.no_grad():
                for imgs in tqdm(loader, desc=s):
                    out.append(dino(pixel_values=imgs.to(device)).last_hidden_state[:, 0, :].cpu().numpy())
            np.save(os.path.join(CLEAN_PROC, f'X_dino_{s}.npy'), np.vstack(out))
        print('DINOv2 : extrait')

X_dino_tr = np.load(os.path.join(CLEAN_PROC, 'X_dino_train.npy'))
print(f'DINOv2 : {X_dino_tr.shape}')

DINOv2 : copie depuis processed/
DINOv2 : (59185, 768)


In [24]:
# ── DINOv2 features — images croppées (split/) ───────────────────────
CROP_DIR = os.path.join(BASE, 'data', 'split')
dst_crop = os.path.join(CLEAN_PROC, 'X_dino_crop_train.npy')

if os.path.exists(dst_crop):
    print('DINOv2 crop : cache trouve')
else:
    from transformers import AutoModel
    from torch.utils.data import Dataset, DataLoader
    import torchvision.transforms as T
    from PIL import Image
    from tqdm import tqdm
    print('DINOv2 crop : extraction (~3 min GPU)...')
    dino = AutoModel.from_pretrained('facebook/dinov2-base').eval().to(device)
    for p in dino.parameters(): p.requires_grad = False
    transform = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

    crop_train = pd.read_csv(os.path.join(CROP_DIR, 'train.csv'))
    crop_val   = pd.read_csv(os.path.join(CROP_DIR, 'val.csv'))
    crop_test  = pd.read_csv(os.path.join(CROP_DIR, 'test.csv'))

    class ImgDS(Dataset):
        def __init__(self, df, img_dir):
            self.df = df.reset_index(drop=True); self.img_dir = img_dir
        def __len__(self): return len(self.df)
        def __getitem__(self, i):
            try: return transform(Image.open(os.path.join(self.img_dir, self.df.iloc[i]['image_name'])).convert('RGB'))
            except: return torch.zeros(3, 224, 224)

    for s, df in [('train', crop_train), ('val', crop_val), ('test', crop_test)]:
        loader = DataLoader(ImgDS(df, os.path.join(CROP_DIR, s)), batch_size=64, shuffle=False, num_workers=0)
        out = []
        with torch.no_grad():
            for imgs in tqdm(loader, desc=f'crop-{s}'):
                out.append(dino(pixel_values=imgs.to(device)).last_hidden_state[:, 0, :].cpu().numpy())
        np.save(os.path.join(CLEAN_PROC, f'X_dino_crop_{s}.npy'), np.vstack(out))
    print('DINOv2 crop : extrait')

X_dino_crop_tr = np.load(os.path.join(CLEAN_PROC, 'X_dino_crop_train.npy'))
print(f'DINOv2 crop : {X_dino_crop_tr.shape}')

DINOv2 crop : extraction (~3 min GPU)...


crop-test: 100%|██████████| 200/200 [05:37<00:00,  1.69s/it]

DINOv2 crop : extrait
DINOv2 crop : (59185, 768)


In [25]:
# ── Features produit : has_desc + zoom_ratio ─────────────────────────
dst_extra = os.path.join(CLEAN_PROC, 'X_extra_train.npy')
if os.path.exists(dst_extra):
    print('X_extra : cache trouve')
else:
    # Charger depuis split_no_crop (memes valeurs dans les deux splits)
    for s, df in [('train', train_split), ('val', val_split), ('test', test_split)]:
        arr = df[['has_desc', 'zoom_ratio']].fillna(0).values.astype('float32')
        np.save(os.path.join(CLEAN_PROC, f'X_extra_{s}.npy'), arr)
    print('X_extra : sauvegarde')

X_extra_tr = np.load(os.path.join(CLEAN_PROC, 'X_extra_train.npy'))
print(f'X_extra (has_desc, zoom_ratio) : {X_extra_tr.shape}')

X_extra : sauvegarde
X_extra (has_desc, zoom_ratio) : (59185, 2)


In [26]:
X_tfidf_tr    = sp.load_npz(os.path.join(CLEAN_PROC, 'X_tfidf_train.npz'))
X_ctfidf_tr   = np.load(os.path.join(CLEAN_PROC, 'X_ctfidf_train.npy'))
X_e5_tr       = np.load(os.path.join(CLEAN_PROC, 'X_e5_train.npy'))
X_dino_tr     = np.load(os.path.join(CLEAN_PROC, 'X_dino_train.npy'))
X_dino_crop_tr= np.load(os.path.join(CLEAN_PROC, 'X_dino_crop_train.npy'))
X_extra_tr    = np.load(os.path.join(CLEAN_PROC, 'X_extra_train.npy'))

print('\n' + '='*60)
print('FEATURES DISPONIBLES — data/processed_clean/')
print('='*60)
print(f'  y splits        : {len(y_train):,} / {len(y_val):,} / {len(y_test):,}')
print(f'  TF-IDF          : {X_tfidf_tr.shape}  sparse')
print(f'  c-TF-IDF        : {X_ctfidf_tr.shape}  dense')
print(f'  e5-large        : {X_e5_tr.shape}  dense')
print(f'  DINOv2 nocrop   : {X_dino_tr.shape}  dense')
print(f'  DINOv2 crop     : {X_dino_crop_tr.shape}  dense')
print(f'  X_extra         : {X_extra_tr.shape}  (has_desc, zoom_ratio)')
print('='*60)
print('Pret pour 02_level1_probas.ipynb')


FEATURES DISPONIBLES — data/processed_clean/
  y splits        : 59,185 / 12,993 / 12,738
  TF-IDF          : (59185, 50000)  sparse
  c-TF-IDF        : (59185, 27)  dense
  e5-large        : (59185, 1024)  dense
  DINOv2 nocrop   : (59185, 768)  dense
  DINOv2 crop     : (59185, 768)  dense
  X_extra         : (59185, 2)  (has_desc, zoom_ratio)
Pret pour 02_level1_probas.ipynb
